# BMIN 5200 — Week 11 in-class exercise
## SHAP and LIME: comparing post-hoc explanations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week11.ipynb)

**Time:** ~25 minutes · **Pairs with:** Explainable AI

### Tasks
- Train a random forest on the Week 9 readmission cohort and read its built-in feature importances
- Compute SHAP values and watch them disagree with those importances about which variable matters most
- Turn one patient's SHAP values into a sentence a discharge nurse could actually act on
- Explain the same patient with LIME, get a different ranking, and confront the fact that neither one is checkable against ground truth

### Background
"The model flagged this patient" is not a clinical communication; "the model flagged this patient because of four prior admissions and a hemoglobin of 9.2" is. Post-hoc explanation methods are how that second sentence gets produced for models nobody can read directly, and they are now routine in FDA submissions and health-system model cards. They are also, as you are about to see, not unique — two defensible methods will give you two different stories about the same prediction, and no experiment in this notebook can settle which is correct.

Setup. `shap` and `lime` are not preinstalled in Colab, so the first cell fetches them; it takes
about twenty seconds. Everything else is standard.

In [ ]:
%pip install -q shap lime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import lime.lime_tabular
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(5200)
pd.set_option("display.width", 120)

## The cohort (regenerated from Week 9)

No file was saved at the end of Week 9. The generator below is the identical function with the
identical seed, so the 1,200 synthetic discharges here are the same 1,200 discharges you ranked
by mutual information two weeks ago, down to the row order. The flaw planted in `capture` is
still there and still undisclosed to the model; Week 12 is where it gets measured.

In [ ]:
def make_readmission_cohort(n_patients=1200, seed=5200):
    """Synthetic 30-day readmission cohort. Not real patient data.

    Identical in the Week 9, Week 11, and Week 12 notebooks — same code, same seed — so all
    three weeks work on exactly the same 1,200 discharges. Nothing is saved to disk between
    notebooks; each one regenerates the cohort inline. The function builds its own generator
    so the cohort does not depend on what else has drawn from `rng` above it.
    """
    rng = np.random.default_rng(seed)

    care_network = rng.choice(["in_network", "community_partner"],
                              size=n_patients, p=[0.68, 0.32])
    partner = care_network == "community_partner"

    age = np.clip(rng.normal(68, 12, n_patients), 40, 95).round(0)
    true_prior_admissions = rng.poisson(1.8, n_patients)
    hemoglobin = np.clip(rng.normal(12.4, 1.6, n_patients)
                         - 0.9 * (true_prior_admissions > 2), 7.0, 17.0)
    creatinine = np.clip(rng.lognormal(np.log(1.0), 0.32, n_patients), 0.4, 6.0)

    # Partner-network patients are likelier to be on Medicaid and to live farther out.
    ins_probs = np.where(partner[:, None],
                         np.array([0.18, 0.34, 0.48]),
                         np.array([0.52, 0.36, 0.12]))
    draw = rng.random(n_patients)
    insurance_type = np.array(["commercial", "medicare", "medicaid"])[
        (draw[:, None] > ins_probs.cumsum(axis=1)).sum(axis=1)]

    distance_from_hospital = np.round(
        np.clip(rng.gamma(2.0, np.where(partner, 9.0, 3.4)), 0.5, 90.0), 1)

    # Ground truth: risk is driven by the TRUE admission history, identically in both groups,
    # and never by distance from the hospital.
    log_odds = (-2.0
                + 0.62 * true_prior_admissions
                + 0.030 * (age - 68)
                - 0.26 * (hemoglobin - 12.4)
                + 0.55 * (creatinine - 1.0))
    risk = 1.0 / (1.0 + np.exp(-log_odds))
    readmitted_30d = (rng.random(n_patients) < risk).astype(int)

    # The planted flaw: only 45% of a partner-network patient's prior admissions reach our
    # chart, against 97% for patients who stay in network. The illness is the same; the
    # documentation is not.
    capture = np.where(partner, 0.45, 0.97)
    recorded_prior = rng.binomial(true_prior_admissions, capture)

    return pd.DataFrame({
        "age": age.astype(int),
        "prior_admissions": recorded_prior,
        "hemoglobin": hemoglobin.round(1),
        "creatinine": creatinine.round(2),
        "insurance_type": insurance_type,
        "distance_from_hospital": distance_from_hospital,
        "care_network": care_network,
        "readmitted_30d": readmitted_30d,
    })


FEATURES = ["age", "prior_admissions", "hemoglobin", "creatinine",
            "insurance_type", "distance_from_hospital", "care_network"]

cohort = make_readmission_cohort()
model_input = cohort[FEATURES].copy()
model_input["insurance_type"] = model_input["insurance_type"].map(
    {"commercial": 0, "medicare": 1, "medicaid": 2})
model_input["care_network"] = model_input["care_network"].map(
    {"in_network": 0, "community_partner": 1})
outcome = cohort["readmitted_30d"]

X_train, X_test, y_train, y_test = train_test_split(
    model_input, outcome, test_size=0.3, random_state=5200, stratify=outcome)

print(f"{len(cohort)} synthetic discharges, {outcome.mean():.1%} readmitted; "
      f"{len(X_train)} train / {len(X_test)} test")

## Part 1 — An opaque model

A random forest of 300 trees, each grown to depth 12, works out to some ninety thousand decision
nodes. Per the "Models Frequently Cited as 'Inherently' Explainable" slide, a single depth-3 tree
qualifies as inherently explainable and this does not — not because it is mysterious, but because
no human will trace 300 trees for one patient. It also barely outperforms the Week 9 tree, which
is worth noticing: we gave up readability for about two points of AUC.

In [ ]:
forest = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=1,
                                random_state=5200, n_jobs=-1)
forest.fit(X_train, y_train)
predicted_risk = forest.predict_proba(X_test)[:, 1]

print(f"Held-out AUC          : {roc_auc_score(y_test, predicted_risk):.3f}")
print(f"Week 9 depth-3 tree   : 0.646 (for comparison)")
print(f"Total decision nodes  : {sum(t.tree_.node_count for t in forest.estimators_):,}")

impurity_importance = pd.Series(forest.feature_importances_, index=FEATURES)
print("\nsklearn's built-in feature_importances_ (mean decrease in Gini impurity):")
print(impurity_importance.sort_values(ascending=False).round(3).to_string())

## Predict before you run

That built-in ranking puts `hemoglobin` and `creatinine` on top and `prior_admissions` well down
the list. But you know the ground truth: the generator makes readmission depend most strongly on
prior admissions, and Week 9's mutual information calculation agreed, giving `prior_admissions`
0.081 bits against 0.028 for hemoglobin.

**Commit to an answer.** When we compute SHAP values in a moment, will SHAP put
`prior_admissions` first, or will it reproduce the built-in ranking? And where do you expect
`distance_from_hospital` — a variable with *no* causal effect on the outcome — to land in each?

## Part 2 — Global importance: SHAP and Gini

SHAP is the additive feature attribution method from the slides: for one prediction, it splits
the gap between the model's average output and this patient's output into one number per feature,
and those numbers are the only attribution satisfying the Shapley axioms. `TreeExplainer` computes
them exactly for tree ensembles in polynomial time, which is why we are using a forest and not a
neural network today. Averaging the absolute values over patients turns local attributions into a
global ranking.

In [ ]:
explainer = shap.TreeExplainer(forest)
shap_values = explainer.shap_values(X_test)[:, :, 1]   # [:, :, 1] = the "readmitted" class

shap_importance = pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURES)

comparison = pd.DataFrame({
    "distinct values": model_input.nunique(),
    "Gini importance": impurity_importance.round(3),
    "Gini rank": impurity_importance.rank(ascending=False).astype(int),
    "mean |SHAP|": shap_importance.round(4),
    "SHAP rank": shap_importance.rank(ascending=False).astype(int),
}).sort_values("SHAP rank")
print(comparison.to_string())

The two methods put a different feature at the top and they disagree about the true driver by
four places. `prior_admissions` — the variable we *built* the outcome from — is ranked fifth of
seven by Gini importance and first by SHAP.

The `distinct values` column is the explanation. Gini importance credits a feature with the total
impurity it removes across every node that splits on it, and a feature with 293 distinct values
offers 292 candidate thresholds at every node, while `prior_admissions` offers seven. Given enough
chances, a continuous variable will find *some* threshold that separates a handful of training
patients by luck, and that lucky split still gets credited. This is the standard high-cardinality
bias of impurity importance, and it is why `distance_from_hospital`, which has no effect on the
outcome whatsoever, outranks the strongest real predictor here.

In [ ]:
order = shap_importance.sort_values().index
positions = np.arange(len(order))

plt.figure(figsize=(8, 4))
plt.barh(positions - 0.2, (impurity_importance[order] / impurity_importance.sum()),
         height=0.4, label="Gini importance (normalized)")
plt.barh(positions + 0.2, (shap_importance[order] / shap_importance.sum()),
         height=0.4, label="mean |SHAP| (normalized)")
plt.yticks(positions, order)
plt.xlabel("share of total importance")
plt.title("Two answers to 'which feature matters most'")
plt.legend()
plt.tight_layout()
plt.show()

The beeswarm below is the plot from the "SHAP visualization" slide. One dot per patient per
feature; horizontal position is that patient's SHAP value, colour is the feature's value. Read
`prior_admissions`: the red dots (many prior admissions) sit far to the right, pushing risk up,
and there is a long right tail. Read `distance_from_hospital`: a dense stripe near zero, meaning
the forest uses it constantly but never learns anything from it.

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False, plot_size=(8, 4))
plt.tight_layout()
plt.show()

## Part 3 — Local explanation for one patient

Global rankings are not what a clinician needs at 4pm on a discharge round. The
"Global vs. Local Feature Importance" slide draws that distinction, and SHAP's local values are
already the per-patient numbers we averaged a moment ago. Patient 214 is an 80-year-old on
Medicaid, seen through the community partner network, whom the forest puts well above a typical
0.35 flagging threshold.

Two TODOs below. Make the sentence say whether each feature raised or lowered the estimate, and
report the three features that moved it most rather than the first three in column order.

In [ ]:
patient_id = 214
row_position = list(X_test.index).index(patient_id)
patient = cohort.loc[patient_id]
patient_shap = shap_values[row_position]
average_risk = explainer.expected_value[1]

print(patient[FEATURES].to_string())
print(f"\nCohort average risk : {average_risk:.3f}")
print(f"This patient        : {predicted_risk[row_position]:.3f}")
print(f"SHAP values sum to  : {average_risk + patient_shap.sum():.3f}  "
      f"(additive attribution: the parts add back up to the whole)\n")


def describe_contribution(feature_name, value, shap_value):
    """One clause a discharge nurse could read without a statistics degree."""
    # TODO: "raised" when shap_value > 0, "lowered" when shap_value < 0.
    direction = "changed"
    return f"{feature_name} of {value} {direction} the estimate by {abs(shap_value):.3f}"


contributions = list(zip(FEATURES, patient[FEATURES].values, patient_shap))
# TODO: sort `contributions` by absolute SHAP value, largest first, before taking three.
top_three = contributions[:3]

print(f"Patient {patient_id}: estimated 30-day readmission risk "
      f"{predicted_risk[row_position]:.0%}, against a cohort average of {average_risk:.0%}.")
print("The model's stated reasons:")
for feature_name, value, shap_value in top_three:
    print("  - " + describe_contribution(feature_name, value, shap_value))

## Part 4 — LIME on the same patient

LIME does something completely different for the same goal. It perturbs this one patient
thousands of times, asks the forest what it thinks of each perturbation, and fits a sparse linear
model to that local cloud — a surrogate that is faithful near this patient and nowhere else. It
never opens the forest. SHAP opens the forest and computes an exact Shapley decomposition. Run it
and compare the rankings side by side.

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=FEATURES,
    class_names=["not readmitted", "readmitted"],
    categorical_features=[FEATURES.index("insurance_type"), FEATURES.index("care_network")],
    random_state=5200)

lime_explanation = lime_explainer.explain_instance(
    X_test.values[row_position], forest.predict_proba, num_features=7, num_samples=3000)

lime_weights = {}
for description, weight in lime_explanation.as_list():
    for feature_name in FEATURES:                 # LIME labels rules, not bare features
        if feature_name in description:
            lime_weights[feature_name] = weight
            break

side_by_side = pd.DataFrame({
    "SHAP value": pd.Series(patient_shap, index=FEATURES).round(4),
    "SHAP rank": pd.Series(np.abs(patient_shap), index=FEATURES).rank(ascending=False).astype(int),
    "LIME weight": pd.Series(lime_weights).round(4),
    "LIME rank": pd.Series(lime_weights).abs().rank(ascending=False).astype(int),
}).sort_values("SHAP rank")
print(f"Patient {patient_id}, explained twice:\n")
print(side_by_side.to_string())

print("\nLIME states its reasons as rules, not features:")
for description, weight in lime_explanation.as_list():
    print(f"  {description:42s} {weight:+.4f}")

The two methods name a different single most important feature for this patient — SHAP says
hemoglobin, LIME says age — and they disagree about the rest of the order too. SHAP ranks
`distance_from_hospital` third, above `prior_admissions`; LIME puts `prior_admissions` second and
distance fourth. Same patient, same forest, same afternoon, two rankings.

Now the uncomfortable part: **there is no experiment in this notebook that settles which one is
right.** We know the data-generating process, so we know distance is causally irrelevant — but
both methods are answering a question about *the model*, not about the disease, and the model
genuinely does use distance. An explanation can be perfectly faithful to a model that is wrong
about the world. That is the gap the "Desiderata beyond prediction accuracy" slide is pointing
at, and it is not closed by picking a better attribution method.

## Carried forward to Week 12

Look at the last two rows of the global comparison table. SHAP finds nonzero attribution for
`insurance_type` and `care_network`, and real weight for `distance_from_hospital`, in a model
predicting who gets a transitional-care nurse. Two of those are administrative facts about a
patient's coverage and geography, not clinical facts about their illness.

Explainability told us the model uses them. It cannot tell us whether that is acceptable, who is
helped and who is harmed, or whether the effect is large enough to matter. Those are fairness
questions and they need different arithmetic. **Week 12 audits this exact model** — same seed,
same forest — and the recording gap planted in Week 9 shows up as a measurable difference in who
gets flagged.

## Discussion

1. If SHAP and LIME disagree about a patient and you must put one sentence in the discharge note,
   how do you choose? Would you show the clinician both, and what would you expect them to do
   with two explanations?
2. `distance_from_hospital` has no causal effect on readmission in the generator, yet both
   methods give it nonzero attribution because the model uses it. Is the explanation wrong, is
   the model wrong, or neither? What would you have to change to fix it?
3. The forest bought us about 0.02 of AUC over a depth-3 tree that a clinician could read
   unaided. At what performance gain would you accept a model that needs SHAP to be interpreted
   at all — and does your answer change if the model triages nurse visits versus withholds them?

## Solutions

Completed versions of the Part 3 TODOs, as markdown so they do not run.

```python
def describe_contribution(feature_name, value, shap_value):
    direction = "raised" if shap_value > 0 else "lowered"
    return f"{feature_name} of {value} {direction} the estimate by {abs(shap_value):.3f}"


contributions = list(zip(FEATURES, patient[FEATURES].values, patient_shap))
contributions.sort(key=lambda item: abs(item[2]), reverse=True)
top_three = contributions[:3]
```

Which prints, for patient 214, something a nurse could read in the hallway:

```
Patient 214: estimated 30-day readmission risk 69%, against a cohort average of 34%.
The model's stated reasons:
  - hemoglobin of 11.5 raised the estimate by 0.119
  - age of 80 raised the estimate by 0.086
  - distance_from_hospital of 12.4 raised the estimate by 0.054
```

Read that third line again, and hold onto it until next week.